# S9 · Observabilidad y costos

**Integrante** · Joel Jonatan Quispe Mamani

**Dimensión** · Latencia y disponibilidad de mediciones

Equipo 04 · AeroStreamPeru · Ciclo 9

## 1. Medición sobre el pipeline real

S8 agrega `isValid`, `processedAt` y `latencyMs` al normalizar mensajes. La consulta de observabilidad guarda evidencia en Parquet y envía métricas a Prometheus por Pushgateway. La latencia mide de la recepción HTTP en el productor a la normalización en Spark. La edad de captura mide desde el tiempo del estado en OpenSky.

In [1]:
from pathlib import Path
import json
from urllib.parse import urlencode
from urllib.request import urlopen

from pyspark.sql import SparkSession, functions as F

PROMETHEUS = "http://prometheus:9090"
ARTIFACTS = Path("/opt/s09-observabilidad-pipeline/artifacts")
EVIDENCIA = ARTIFACTS / "observaciones"
CADENCIA_SEGUNDOS = 120
CAPTURAS_MINIMAS = 4
if not EVIDENCIA.is_dir():
    raise FileNotFoundError("Ejecuta el flujo con --observabilidad y consulta OpenSky como indica el README de S9.")

spark = (
    SparkSession.builder.master("local[2]").appName("AeroStreamPeru-S09-Evidencia")
    .config("spark.driver.memory", "512m")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
observaciones = spark.read.parquet(str(EVIDENCIA)).cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 20:16:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 20:16:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

## 2. Prometheus, Grafana y lastProgress

El atraso de Spark se obtiene de los offsets que Spark reporta, porque esta consulta guarda su progreso en el checkpoint. El target de Kafka Exporter refleja disponibilidad del exportador. Consulta el [tablero de Grafana](http://localhost:43000/d/aerostreamperu-s09/aerostreamperu-observabilidad) para correlacionar las señales.

In [2]:
def api_prometheus(ruta, parametros=None):
    url = f"{PROMETHEUS}/api/v1/{ruta}"
    if parametros:
        url += f"?{urlencode(parametros)}"
    with urlopen(url, timeout=10) as respuesta:
        resultado = json.load(respuesta)
    if resultado["status"] != "success":
        raise RuntimeError(f"Falló la consulta a Prometheus: {ruta}")
    return resultado["data"]


activos = api_prometheus("targets")["activeTargets"]
print("Targets:", [(entrada["labels"]["job"], entrada["health"]) for entrada in activos])

consultas = {
    "Kafka Exporter": 'up{job="kafka-exporter"}',
    "Pico observado de entrada de Spark": 'max_over_time(aerostream_spark_entrada_mensajes_por_segundo{job="aerostream_spark_deduplicacion"}[30m])',
    "Procesados por segundo en el último lote": 'aerostream_spark_procesado_mensajes_por_segundo{job="aerostream_spark_deduplicacion"}',
    "Retraso de Spark en mensajes": 'aerostream_spark_retraso_kafka_mensajes{job="aerostream_spark_deduplicacion"}',
    "Latencia media del último lote (s)": "aerostream_telemetria_latencia_media_segundos",
    "Capturas faltantes en la última ventana": "aerostream_ventana_capturas_faltantes",
}
resultados = {}
for nombre, expresion in consultas.items():
    datos = api_prometheus("query", {"query": expresion})["result"]
    valores = [float(fila["value"][1]) for fila in datos]
    resultados[nombre] = max(valores) if valores else None
    print(f"{nombre}: {resultados[nombre]}")

with (ARTIFACTS / "progreso_deduplicacion.json").open(encoding="utf-8") as archivo:
    progreso = json.load(archivo)
print({
    campo: progreso[campo] for campo in ("batchId", "numInputRows", "inputRowsPerSecond", "processedRowsPerSecond")
})

Targets: [('kafka-exporter', 'up'), ('prometheus', 'up'), ('pushgateway', 'up')]
Kafka Exporter: 1.0
Pico observado de entrada de Spark: 3.0
Procesados por segundo en el último lote: 4.4
Retraso de Spark en mensajes: 0.0
Latencia media del último lote (s): 3.843
Capturas faltantes en la última ventana: 0.0
{'batchId': 114, 'numInputRows': 13, 'inputRowsPerSecond': 2.6, 'processedRowsPerSecond': 3.0}


## 3. Latencia agregada y evidencia individual

Agrupamos la latencia de estados con medición disponible por minuto UTC de procesamiento.

In [3]:
telemetria = observaciones.filter((F.col("tipo") == "estado") & F.col("latencyMs").isNotNull())
telemetria.select("time", "isValid", "processedAt", "latencyMs").orderBy("processedAt").show(8, truncate=False)
latencia_por_minuto = (
    telemetria.groupBy(F.window("processedAt", "1 minute").alias("intervalo"))
    .agg(
        F.count("*").alias("eventos"),
        F.round(F.avg("latencyMs"), 2).alias("media_ms"),
        F.percentile_approx("latencyMs", 0.95).alias("p95_ms"),
    )
    .select(F.col("intervalo.start").alias("minuto_utc"), "eventos", "media_ms", "p95_ms")
    .orderBy("minuto_utc")
)
latencia_por_minuto.show(truncate=False)

+----------+-------+-----------------------+---------+
|time      |isValid|processedAt            |latencyMs|
+----------+-------+-----------------------+---------+
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542000|true   |2026-11-24 17:42:05.017|3393.0   |
|1795542120|true   |2026-11-24 17:42:15.012|3658.0   |
|1795542120|true   |2026-11-24 17:42:15.012|3658.0   |
+----------+-------+-----------------------+---------+
only showing top 8 rows


[Stage 3:===================>                                       (2 + 2) / 6]

+-------------------+-------+---------+--------+
|minuto_utc         |eventos|media_ms |p95_ms  |
+-------------------+-------+---------+--------+
|2026-11-24 17:42:00|12     |3525.5   |3658.0  |
|2026-11-24 18:01:00|15     |104145.73|105198.0|
|2026-11-24 18:29:00|15     |111509.13|146105.0|
|2026-11-24 18:30:00|7      |3861.0   |3861.0  |
|2026-11-24 18:32:00|8      |3916.0   |3916.0  |
|2026-11-24 18:34:00|9      |3892.0   |3892.0  |
|2026-11-24 18:36:00|10     |3880.0   |3880.0  |
|2026-11-24 18:38:00|8      |3941.0   |3941.0  |
|2026-11-24 18:40:00|11     |3575.0   |3575.0  |
|2026-11-24 18:42:00|12     |3868.0   |3868.0  |
|2026-11-24 18:44:00|15     |3890.0   |3890.0  |
|2026-11-24 18:46:00|16     |3846.0   |3846.0  |
|2026-11-24 18:48:00|19     |3605.0   |3605.0  |
|2026-11-24 18:50:00|19     |3799.0   |3799.0  |
|2026-11-24 18:52:00|18     |3934.0   |3934.0  |
|2026-11-24 18:54:00|19     |3918.0   |3918.0  |
|2026-11-24 18:56:00|17     |3194.0   |3194.0  |
|2026-11-24 18:58:00

In [4]:
observaciones.filter(F.col("tipo") == "captura").select(
    "time", "estado", "isValid", "estados_publicados"
).orderBy("time", ascending=False).show(8, truncate=False)

+----------+------+-------+------------------+
|time      |estado|isValid|estados_publicados|
+----------+------+-------+------------------+
|1795551240|ok    |true   |13                |
|1795551120|ok    |true   |13                |
|1795551000|ok    |true   |11                |
|1795550880|ok    |true   |12                |
|1795550760|ok    |true   |9                 |
|1795550640|ok    |true   |12                |
|1795550520|ok    |true   |10                |
|1795550400|ok    |true   |12                |
+----------+------+-------+------------------+
only showing top 8 rows


## 4. Alertas propuestas

Los umbrales se calculan con el tamaño máximo de la captura y el P95 observados. La cobertura mínima se toma del contrato de los modelos y el criterio de persistencia de una alerta se expresa en función de la cadencia de dos minutos.

In [5]:
capturas_ok = (
    observaciones.filter((F.col("tipo") == "captura") & F.col("isValid"))
    .select("time", "estados_publicados").distinct()
)
muestra = capturas_ok.agg(
    F.count("*").alias("capturas"),
    F.avg("estados_publicados").alias("media"),
    F.max("estados_publicados").alias("maximo"),
).first()
latencia = telemetria.agg(
    F.count("*").alias("eventos"),
    F.percentile_approx("latencyMs", 0.95).alias("p95_ms"),
).first()
assert muestra.capturas > 0 and latencia.eventos > 0, "Se necesitan capturas reales con latencia medida."

umbral_atraso_mensajes = 2 * (muestra.maximo + 1)
umbral_latencia_segundos = max(10, 2 * latencia.p95_ms / 1000)
print(f"Base medida: {muestra.capturas} capturas, máximo {muestra.maximo} estados por captura, P95 {latencia.p95_ms:.1f} ms.")
print(f"1. Atraso Spark > {umbral_atraso_mensajes} mensajes durante {CADENCIA_SEGUNDOS} s: equivale a más de dos capturas de tamaño máximo observado, con su confirmación.")
print(f"2. Latencia P95 > {umbral_latencia_segundos:.2f} s durante {CADENCIA_SEGUNDOS} s: se usa el doble del P95 medido, con un margen mínimo de dos disparos de Spark.")
print(f"3. Cobertura de una ventana cerrada < {CAPTURAS_MINIMAS}/5: revisar la fuente y marcarla como insuficiente para inferencia.")

Base medida: 57 capturas, máximo 19 estados por captura, P95 3997.0 ms.
1. Atraso Spark > 40 mensajes durante 120 s: equivale a más de dos capturas de tamaño máximo observado, con su confirmación.
2. Latencia P95 > 10.00 s durante 120 s: se usa el doble del P95 medido, con un margen mínimo de dos disparos de Spark.
3. Cobertura de una ventana cerrada < 4/5: revisar la fuente y marcarla como insuficiente para inferencia.


## 5. Costos de referencia y escalado

Escenario de tres máquinas Basic de 2 vCPU y 4 GiB. Una para Kafka y el productor, otra para Spark y otra para Prometheus y Grafana. El almacenamiento se proyecta con la media de estados por captura, una confirmación por captura, seis ventanas por hora y los bytes por fila medidos en Parquet. Los lotes pequeños tienen sobrecostos de metadatos. La tarifa es una referencia de planificación para 30 días de retención. [Precios de máquinas](https://www.digitalocean.com/pricing/droplets) · [Precios de almacenamiento](https://docs.digitalocean.com/products/volumes/details/pricing/)

In [6]:
from math import ceil

GIB = 1024 ** 3
RETENCION_DIAS = 30
USD_DROPLET_MES = 24.0
USD_DROPLET_HORA = 0.03571
USD_GIB_MES = 0.10
archivos = list(EVIDENCIA.rglob("*.parquet"))
bytes_archivos = sum(ruta.stat().st_size for ruta in archivos)
filas = observaciones.count()
bytes_por_fila = bytes_archivos / filas
filas_por_hora = (3600 / CADENCIA_SEGUNDOS) * (muestra.media + 1) + 6
gib_por_hora = bytes_por_fila * filas_por_hora / GIB
gib_retenidos = gib_por_hora * 24 * RETENCION_DIAS

costo_computo = 3 * USD_DROPLET_MES
costo_volumen = max(1, ceil(gib_retenidos)) * USD_GIB_MES

print(f"Muestra: {filas} filas, {bytes_archivos:,} bytes Parquet, media {muestra.media:.2f} estados por captura.")
print(f"Pico observado de entrada: {resultados['Pico observado de entrada de Spark']} mensajes/s.")
print(f"Crecimiento estimado: {gib_por_hora:.6f} GiB/h; {gib_retenidos:.3f} GiB con {RETENCION_DIAS} días.")
print(f"Cómputo: 3 × USD {USD_DROPLET_HORA:.5f}/h o USD {costo_computo:.2f}/mes con los topes del plan.")
print(f"Volumen adicional opcional: USD {costo_volumen:.2f}/mes.")
print(f"Total de referencia con volumen adicional: USD {costo_computo + costo_volumen:.2f}/mes.")

Muestra: 766 filas, 532,776 bytes Parquet, media 12.14 estados por captura.
Pico observado de entrada: 3.0 mensajes/s.
Crecimiento estimado: 0.000259 GiB/h; 0.187 GiB con 30 días.
Cómputo: 3 × USD 0.03571/h o USD 72.00/mes con los topes del plan.
Volumen adicional opcional: USD 0.10/mes.
Total de referencia con volumen adicional: USD 72.10/mes.


## 6. Interpretación y plan de escalado

Si sube el P95, reviso primero cuándo recibió el productor cada captura y cuánto tardó Spark en procesarla. Un aumento persistente junto al atraso del consumidor apunta a revisar el procesamiento. Para mi indicador también importa si las ventanas reúnen suficientes capturas válidas. Los umbrales de atraso y latencia indican cuándo revisar Spark. La cobertura indica cuándo revisar la fuente. La prueba es corta y sirve como línea base inicial. Para el informe individual, adjunta tus capturas de Prometheus, Grafana y las salidas del notebook con reloj y perfil visibles.

In [7]:
spark.stop()